# Regresión lineal múltiple usando lm()

*¿Dónde aparecen en la salida de R los resultados que calculamos con matrices?*

Estimaremos el mismo modelo con `lm()`, extraeremos sus resultados y calcularemos predicciones para nuevas combinaciones de precio y publicidad. Este notebook puede ejecutarse de forma independiente.

## Importamos los datos

Usamos la base de Big Andy's Burger Barn de Hill, Griffiths y Lim (2011): 75 observaciones de franquicias en distintas ciudades. `sales` representa los ingresos mensuales por ventas y `advert` el gasto mensual en publicidad, ambos en miles de dólares. `price` es un índice de precios de los productos, expresado en dólares.

La ruta es relativa a la carpeta del notebook. Mostramos las primeras observaciones.

In [1]:
library(readr)

datos <- read_csv("../data/andy.csv")
head(datos)

Rows: 75 Columns: 3


── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (3): sales, price, advert



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


sales,price,advert
<dbl>,<dbl>,<dbl>
73.2,5.69,1.3
71.8,6.49,2.9
62.4,5.63,0.8
67.4,6.22,0.7
89.3,5.02,1.5
70.3,6.41,1.3


## Planteamos el modelo

Queremos estimar

$$
sales_i=\beta_1+\beta_2price_i+\beta_3advert_i+u_i.
$$

Cada pendiente representa una relación parcial: el cambio en las ventas esperadas asociado con un cambio en un regresor, manteniendo constante el otro.

## Estimamos el modelo

En la fórmula, `sales ~ price + advert` indica que explicamos las ventas mediante precio y publicidad. `lm()` incluye una constante por defecto.

In [2]:
mco <- lm(sales ~ price + advert, data = datos)
coef(mco)

(Intercept)       price      advert 
 118.913610   -7.907854    1.862584

## Examinamos el resumen

`summary()` reúne los coeficientes, sus errores estándar y la bondad de ajuste. Guardamos el resultado en `smco` para extraer información después.

Por ahora nos concentramos en `Estimate`, `Std. Error`, `Residual standard error`, `Multiple R-squared` y `Adjusted R-squared`. Las columnas de pruebas y la prueba F se estudiarán en el capítulo de inferencia.

In [3]:
smco <- summary(mco)
smco


Call:
lm(formula = sales ~ price + advert, data = datos)

Residuals:
     Min       1Q   Median       3Q      Max 
-13.4825  -3.1434  -0.3456   2.8754  11.3049 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 118.9136     6.3516  18.722  < 2e-16 ***
price        -7.9079     1.0960  -7.215 4.42e-10 ***
advert        1.8626     0.6832   2.726  0.00804 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 4.886 on 72 degrees of freedom
Multiple R-squared:  0.4483,	Adjusted R-squared:  0.4329 
F-statistic: 29.25 on 2 and 72 DF,  p-value: 5.041e-10


## Interpretamos los coeficientes

La pendiente de `price` compara las ventas esperadas para distintos precios manteniendo fija la publicidad. La pendiente de `advert` compara las ventas esperadas para distintos gastos en publicidad manteniendo fijo el precio.

`sales` y `advert` están en miles de dólares. Un aumento de una unidad en `advert` significa mil dólares adicionales de publicidad. La constante corresponde a precio y publicidad iguales a cero, una combinación fuera de los valores observados en esta base.

## Matriz de varianzas y covarianzas

`vcov()` devuelve la misma matriz que calculamos como $s^2(\mathbf X^\top\mathbf X)^{-1}$. Las raíces cuadradas de su diagonal coinciden con la columna `Std. Error` del resumen.

In [4]:
Vb <- vcov(mco)
Vb
sqrt(diag(Vb))

,(Intercept),price,advert
(Intercept),40.3432990,-6.79506412,-0.74842060
price,-6.7950641,1.20120070,-0.01974215
advert,-0.7484206,-0.01974215,0.46675606


(Intercept)       price      advert 
  6.3516375   1.0959930   0.6831955

## Valores ajustados y residuos

`fitted()` obtiene los valores ajustados para las observaciones de la muestra y `residuals()` obtiene sus residuos.

In [5]:
yhat <- fitted(mco)
e <- residuals(mco)
head(data.frame(Observado = datos$sales, Ajustado = yhat, Residuo = e))

,Observado,Ajustado,Residuo
,<dbl>,<dbl>,<dbl>
1,73.2,76.33928,-3.1392788
2,71.8,72.99313,-1.1931302
3,62.4,75.88246,-13.4824580
4,67.4,71.03057,-3.6305655
5,89.3,82.01006,7.2899419
6,70.3,70.64562,-0.3456237


## Recuperamos $RSS$, $s^2$ y los grados de libertad

`df.residual()` devuelve $n-k$. El valor `smco$sigma` es la desviación estándar estimada de los errores, $s$, por lo que su cuadrado es $s^2$.

In [6]:
RSS <- sum(e^2)
gl <- df.residual(mco)
s2 <- smco$sigma^2
c(RSS = RSS, gl = gl, s2 = s2)

RSS         gl         s2 
1718.94294   72.00000   23.87421

## Recuperamos $R^2$ y $R^2$ ajustada

Estos componentes de `smco` corresponden a las medidas que calculamos en el notebook matricial.

In [7]:
c(R2 = smco$r.squared, R2_ajustada = smco$adj.r.squared)

R2 R2_ajustada 
  0.4482578   0.4329316

## Tabla ANOVA

`anova(mco)` muestra una descomposición secuencial de la variación. `price` entra primero, después de la constante; `advert` entra después de `price`. La columna `Sum Sq` indica cuánto disminuye RSS al incorporar cada término. La fila `Residuals` contiene el RSS del modelo completo.

La suma de los valores de `Sum Sq`, incluida la fila de residuos, corresponde a TSS. `Mean Sq` en la fila de residuos es $s^2$. Retomaremos `F value` y `Pr(>F)` al estudiar inferencia.

In [8]:
tabla_anova <- anova(mco)
tabla_anova

,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<int>,<dbl>,<dbl>,<dbl>,<dbl>
price,1,1219.0910,1219.09103,51.06310,5.945932e-10
advert,1,177.4479,177.44790,7.43262,8.038182e-03
Residuals,72,1718.9429,23.87421,NA,NA


## ¿Qué ocurre si cambiamos el orden?

Invertimos el orden en la fórmula: ahora `advert` entra primero y `price` después. La asignación de las sumas de cuadrados a los regresores puede cambiar, aunque los coeficientes del modelo completo, sus valores ajustados y su RSS son los mismos, salvo diferencias de redondeo.

Por eso, las pruebas de las filas de esta tabla no deben interpretarse automáticamente como las pruebas individuales del resumen: cada fila corresponde al término añadido después de los que la preceden.

In [9]:
mco_orden <- lm(sales ~ advert + price, data = datos)
anova(mco_orden)

,Df,Sum Sq,Mean Sq,F value,Pr(>F)
,<int>,<dbl>,<dbl>,<dbl>,<dbl>
advert,1,153.6546,153.65462,6.436009,1.335043e-02
price,1,1242.8843,1242.88431,52.059710,4.423997e-10
Residuals,72,1718.9429,23.87421,NA,NA


## Preparamos nuevas combinaciones

Usaremos tres escenarios: el primero reproduce el ejemplo del libro; el segundo aumenta la publicidad con el precio fijo; el tercero aumenta el precio con la publicidad fija. Los valores están dentro de los rangos observados de cada regresor.

`newdata` debe contener los regresores con los mismos nombres que usamos en el modelo. Cada fila representa una combinación de precio y publicidad.

In [10]:
newx <- data.frame(
  price = c(5.5, 5.5, 6.0),
  advert = c(1.2, 2.2, 1.2)
)
newx

price,advert
<dbl>,<dbl>
5.5,1.2
5.5,2.2
6.0,1.2


## Calculamos las predicciones

`predict()` evalúa la ecuación estimada en cada fila de `newx`:

$$
\hat y_0=b_1+b_2price_0+b_3advert_0.
$$

Son predicciones puntuales en miles de dólares; no garantizan las ventas que se observarán.

In [11]:
pred <- predict(mco, newdata = newx)
data.frame(newx, Ventas_predichas = pred)

,price,advert,Ventas_predichas
,<dbl>,<dbl>,<dbl>
1,5.5,1.2,77.65551
2,5.5,2.2,79.51810
3,6.0,1.2,73.70159


## Interpretación

Para un precio de 5.50 dólares y un gasto publicitario de 1,200 dólares, las ventas predichas son aproximadamente 77.656 miles de dólares, es decir, 77,656 dólares.

Compara el segundo escenario con el primero: al aumentar la publicidad en mil dólares y conservar el precio, la predicción aumenta en aproximadamente 1.863 miles de dólares. Compara el tercero con el primero: al aumentar el precio en 0.50 dólares y conservar la publicidad, la predicción disminuye en aproximadamente 3.954 miles de dólares.

Estas diferencias muestran la interpretación de las pendientes como relaciones parciales. Conviene usar el modelo para combinaciones comparables con las observadas; los coeficientes no justifican extrapolaciones arbitrarias.

## Documentación

- [Tabla ANOVA para modelos lineales en R](https://stat.ethz.ch/R-manual/R-patched/library/stats/html/anova.lm.html)
- [Predicciones de modelos lineales en R](https://stat.ethz.ch/R-manual/R-patched/library/stats/html/predict.lm.html)